# MMAction2 fine-tuned refinement v3

Notebook này giữ checkpoint fine-tuned hiện có ở Drive, chạy nó trên AIRTLab + SBU và train một head hiệu chỉnh riêng cho các tình huống hai người đến gần, đứng gần, ôm, bắt tay và high-five. Backbone AVA, detector và head fine-tuned cũ đều được khóa.

Cần có hai file trên Drive:
- `SENTINEL/artifacts/mmaction_indoor_review_v2/indoor_review_pack.zip`
- `SENTINEL/artifacts/mmaction_interaction_v1/mmaction_interaction_pilot.zip`


In [ ]:
import time, threading
from pathlib import Path
from google.colab import drive, runtime
import torch
assert torch.cuda.is_available(), 'Hãy chọn GPU trong Runtime > Change runtime type.'
DEADLINE=time.monotonic()+45*60
watchdog=threading.Timer(45*60, runtime.unassign); watchdog.daemon=True; watchdog.start()
drive.mount('/content/drive', timeout_ms=180000)
ROOT=Path('/content/drive/MyDrive/SENTINEL')
def remaining(reserve=0):
    seconds=DEADLINE-time.monotonic()-reserve
    if seconds<=0: raise TimeoutError('Hết ngân sách GPU; file trên Drive giữ nguyên.')
    return seconds
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import json, os, shutil, subprocess, sys, urllib.request, zipfile
with urllib.request.urlopen('https://api.github.com/repos/TranDuon/anomaly-behavior-detector-system/commits/main', timeout=60) as response: REVISION=json.load(response)['sha']
source_zip=Path('/content/sentinel-source.zip')
with urllib.request.urlopen('https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/'+REVISION, timeout=90) as source, source_zip.open('wb') as target: shutil.copyfileobj(source,target)
def extract_safe(archive, destination):
    with zipfile.ZipFile(archive) as package:
        assert all((destination/item.filename).resolve().is_relative_to(destination.resolve()) for item in package.infolist())
        package.extractall(destination)
workspace=Path('/content/sentinel-source'); workspace.mkdir(exist_ok=True); extract_safe(source_zip,workspace)
CODE=workspace/('anomaly-behavior-detector-system-'+REVISION); DEMO=CODE/'mmaction2_demo'; MANIFEST=CODE/'benchmarks/mmaction_interaction_indoor_v2.json'
manifest=json.loads(MANIFEST.read_text(encoding='utf-8')); assert manifest.get('readiness')=='ready', manifest.get('readiness_reasons')
os.chdir(CODE); sys.path.insert(0,str(CODE)); from scripts import colab_process
PY='/content/mmaction-env/bin/python'
def command(args, log, cap=600): return colab_process.run_logged(args, RUN/log, min(cap,remaining(180)),revision=REVISION)
subprocess.run([sys.executable,'-m','pip','install','-q','uv==0.8.22'],check=True)
subprocess.run(['uv','python','install','3.10'],check=True)
subprocess.run(['uv','venv','--python','3.10','--seed','/content/mmaction-env'],check=True)
subprocess.run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'],check=True)
subprocess.run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv','-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html','mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops','importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'],check=True)
subprocess.run(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',str(DEMO/'repo')],check=True)
subprocess.run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',str(DEMO/'repo')],check=True)


In [ ]:
import hashlib
PACK=Path('/content/indoor-pack'); CACHE=Path('/content/tuned-cache-v3'); WEIGHTS=Path('/content/tuned-weights-v3')
SHARED=ROOT/'artifacts/mmaction_tuned_refinement_v3'; RUN=SHARED/'runs'/REVISION
for folder in (PACK,CACHE,WEIGHTS,SHARED,RUN): folder.mkdir(parents=True,exist_ok=True)
package=ROOT/'artifacts/mmaction_indoor_review_v2/indoor_review_pack.zip'; legacy=ROOT/'artifacts/mmaction_interaction_v1/mmaction_interaction_pilot.zip'
assert package.is_file(), 'Thiếu indoor_review_pack.zip'
assert legacy.is_file(), 'Thiếu mmaction_interaction_pilot.zip của checkpoint fine-tuned cũ'
extract_safe(package,PACK); extract_safe(legacy,WEIGHTS)
if (SHARED/'feature_cache.zip').is_file(): extract_safe(SHARED/'feature_cache.zip',CACHE)
checks={'action_slowonly_r101.pth':('https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth','163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),'person_faster_rcnn_r50.pth':('https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth','a5d8aa15c75b961b9c73e28a6e92af370835639a2939ff890852b052a5256a5b')}
for name,(url,expected) in checks.items():
    target=WEIGHTS/name
    if not target.is_file():
        with urllib.request.urlopen(url,timeout=120) as source, target.open('wb') as destination: shutil.copyfileobj(source,destination)
    assert hashlib.sha256(target.read_bytes()).hexdigest()==expected, name+' SHA mismatch'
os.environ.update(PYTHONPATH=str(CODE)+os.pathsep+str(DEMO)+os.pathsep+str(DEMO/'repo'),CUDA_MODULE_LOADING='LAZY',OMP_NUM_THREADS='4')
minutes=min(30,remaining(180)/60); assert minutes>=1
try:
    colab_process.run_logged([PY,str(DEMO/'train_tuned_refinement_v3.py'),'--manifest',str(MANIFEST),'--pack',str(PACK),'--cache',str(CACHE),'--weights',str(WEIGHTS),'--output',str(RUN),'--minutes',str(minutes),'--epochs','120'],RUN/'train.log',min(minutes*60+20,remaining(150)),revision=REVISION)
    print(json.dumps(json.loads((RUN/'report.json').read_text()),ensure_ascii=False,indent=2)); print('Gói:',RUN/'mmaction_tuned_refinement_v3_pilot.zip')
finally:
    partial=SHARED/'feature_cache.partial.zip'
    with zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED) as archive:
        for path in CACHE.glob('*.npz'): archive.write(path,path.name)
    partial.replace(SHARED/'feature_cache.zip'); runtime.unassign()
